# 04 Attribution graphs (preregistration 7.3)

200 evaluation prompts (the first 100 carry and the first 100 no-carry of `q1.eval_pairs()`). One graph per prompt, summarized by `helixadd.graphs.summarize` (influence share per group, edges between the two primary groups) and saved row by row to `results/q1_graphs/summary.csv`, so a new session resumes. About an hour on a T4.

Setup: Accelerator GPU T4, Internet on, secret `HF_TOKEN`. Run as Save Version, Save & Run All.

In [ ]:
!rm -rf helix_addition && git clone https://github.com/amark-23/helix_addition.git
!pip install -q git+https://github.com/safety-research/circuit-tracer@v0.5.3 pytest

In [ ]:
import os, sys
os.chdir("/kaggle/working/helix_addition")
sys.path.insert(0, "/kaggle/working/helix_addition/src")
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
login(UserSecretsClient().get_secret("HF_TOKEN"))

In [ ]:
from helixadd import model as hmodel, q1, prompts, graphs
from helixadd.groups import load_labels
model = hmodel.load_model()

In [ ]:
import pandas as pd
from pathlib import Path
from circuit_tracer import attribute

pairs = q1.eval_pairs()
chosen = [p for p in pairs if prompts.is_carry(*p)][:100] + [p for p in pairs if not prompts.is_carry(*p)][:100]
categories = graphs.feature_categories(load_labels())
out = Path("results/q1_graphs"); out.mkdir(parents=True, exist_ok=True)
print(len(chosen), "prompts")

## One graph first: check the summary

In [ ]:
import time
t = time.time()
g = attribute(prompts.make_prompt(*chosen[0]), model, offload=None, verbose=False)
row = graphs.summarize(g, categories)
print(f"{time.time() - t:.1f} s")
print({k: round(v, 3) for k, v in row.items()})
print("influence shares sum to", round(sum(v for k, v in row.items() if k.startswith("influence")), 4))

## All 200

In [ ]:
import torch
f = out / "summary.csv"
done = set(map(tuple, pd.read_csv(f)[["a", "b"]].values)) if f.exists() else set()
for i, (a, b) in enumerate(chosen):
    if (a, b) in done:
        continue
    g = attribute(prompts.make_prompt(a, b), model, offload=None, verbose=False)
    row = {"a": a, "b": b, "carry": prompts.is_carry(a, b), **graphs.summarize(g, categories)}
    pd.DataFrame([row]).to_csv(f, mode="a", header=not f.exists(), index=False)
    del g; torch.cuda.empty_cache()
    if i % 20 == 0:
        print(i, "of", len(chosen))
print("done")